In [8]:
import os
import seaborn as sns

titanic = sns.load_dataset("titanic")
propre = titanic.copy()

In [9]:
manquants_avant = propre.isna().sum()
manquants_avant[manquants_avant > 0]

age            177
embarked         2
deck           688
embark_town      2
dtype: int64

In [10]:
nb_doublons = propre.duplicated().sum()
nb_doublons

np.int64(107)

In [11]:
mediane_age = propre.groupby(["class", "sex"], observed=True)["age"].transform("median")
propre["age"] = propre["age"].fillna(mediane_age)

propre = propre.dropna(subset=["embark_town"])
propre = propre.drop(columns=["deck"])
propre.isna().sum()

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64

In [12]:
propre = propre.drop(columns=["alive", "pclass", "embarked"])

propre["sex"] = propre["sex"].astype("category")
propre["embark_town"] = propre["embark_town"].astype("category")
propre.dtypes

survived          int64
sex            category
age             float64
sibsp             int64
parch             int64
fare            float64
class          category
who                 str
adult_male         bool
embark_town    category
alone              bool
dtype: object

In [13]:
q1 = propre["fare"].quantile(0.25)
q3 = propre["fare"].quantile(0.75)
borne_haute = q3 + 1.5 * (q3 - q1)

billets_hors_bornes = propre[propre["fare"] > borne_haute]
nb_billets_hors_bornes = len(billets_hors_bornes)
billets_gratuits = (propre["fare"] == 0).sum()

print(f"Borne haute : {borne_haute:.2f}")
print(f"Billets au-dessus : {nb_billets_hors_bornes}")
print(f"Billets à 0 : {billets_gratuits}")

Borne haute : 65.66
Billets au-dessus : 114
Billets à 0 : 15


In [14]:
os.makedirs("data", exist_ok=True)
propre.to_csv("data/titanic_propre.csv", index=False)


| Quoi | Pourquoi | Lignes |
|---|---|---|
| deck supprimée | 77 % de manquants, inexploitable | 891 |
| age imputé | médiane de la classe et du sexe, pour ne pas fausser les groupes | 177 |
| embark_town : 2 passagers supprimés | port inconnu, impossible de le deviner | 2 |
| doublons conservés | pas d'identifiant passager, deux lignes identiques peuvent être deux personnes | 107 |
| alive, pclass, embarked supprimées | doublons de survived, class et embark_town | 889 |
| sex, embark_town en category | peu de valeurs possibles | 889 |
| billets chers conservés | vrais billets de 1re classe, pas des erreurs | 116 |
| billets à 0 conservés | probablement équipage ou invités, à signaler | 15 |